# 📄 Notebook 01 — Document Preprocessing & Index Building
**Amlgo Labs RAG Chatbot Assignment**

This notebook covers:
1. Loading and cleaning `policies.pdf`
2. Sentence-aware chunking analysis
3. Embedding with `bge-small-en-v1.5`
4. Building the ChromaDB vector index
5. Visualizing chunk distribution

In [ ]:
import sys
sys.path.append('..')

import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from config import DEFAULT_DOCUMENT, CHUNK_SIZE_CHARS, CHUNK_OVERLAP_CHARS
from src.document_loader import load_pdf
from src.chunker import chunk_text, save_chunks
from src.vectorstore import build_index, get_chunk_count

sns.set_theme(style='darkgrid', palette='muted')
print('✅ Imports successful')

## Step 1 — Load & Clean Document

In [ ]:
raw_text = load_pdf(DEFAULT_DOCUMENT)
print(f'Document: {DEFAULT_DOCUMENT.name}')
print(f'Total characters: {len(raw_text):,}')
print(f'Total words: {len(raw_text.split()):,}')
print(f'\n--- First 600 chars ---')
print(raw_text[:600])

## Step 2 — Chunking Analysis

In [ ]:
chunks = chunk_text(raw_text, source_name=DEFAULT_DOCUMENT.name)
df = pd.DataFrame(chunks)

print(f'Total chunks: {len(chunks)}')
print(f'\nWord count stats:')
print(df['word_count'].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Chunk Distribution Analysis', fontsize=14, fontweight='bold')

# Word count histogram
axes[0].hist(df['word_count'], bins=20, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(df['word_count'].mean(), color='red', linestyle='--', label=f"Mean: {df['word_count'].mean():.0f}")
axes[0].set_title('Word Count per Chunk')
axes[0].set_xlabel('Words')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Char count histogram
axes[1].hist(df['char_count'], bins=20, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(df['char_count'].mean(), color='navy', linestyle='--', label=f"Mean: {df['char_count'].mean():.0f}")
axes[1].set_title('Character Count per Chunk')
axes[1].set_xlabel('Characters')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('../chunks/chunk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chunks/chunk_distribution.png')

## Step 3 — Save Chunks

In [ ]:
save_chunks(chunks)
print(f'Saved {len(chunks)} chunks to chunks/chunks.json')
print('\nSample chunk (chunk_0010):')
print('-' * 60)
if len(chunks) > 10:
    print(chunks[10]['text'])
    print(f'\nWord count: {chunks[10]["word_count"]}')

## Step 4 — Build ChromaDB Index

In [ ]:
print('Building vector index...')
build_index(chunks)
count = get_chunk_count()
print(f'\n✅ Index ready: {count} chunks stored in ChromaDB')

## Step 5 — Verify Retrieval

In [ ]:
from src.retriever import retrieve

test_query = 'What personal data is collected from users?'
results = retrieve(test_query, k=3)

print(f'Query: "{test_query}"')
print(f'Retrieved {len(results)} chunks:\n')
for r in results:
    print(f"[{r['score']:.3f}] {r['chunk_id']}")
    print(f"{r['text'][:200]}...")
    print()